In [ ]:
!pip uninstall sqlalchemy==1.4 -y && pip install sqlalchemy==1.3.24

In [ ]:
!pip install ipython-sql
!pip install ipython-sql prettytable

In [ ]:
%load_ext sql

In [ ]:
import csv, sqlite3
import prettytable
prettytable.DEFAULT = 'DEFAULT'

con = sqlite3.connect("my_data1.db")
cur = con.cursor()

In [ ]:
!pip install -q pandas

In [ ]:
%sql sqlite:///my_data1.db

In [ ]:
import pandas as pd
df = pd.read_csv("https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_2/data/Spacex.csv")
df.to_sql("SPACEXTBL", con, if_exists='replace', index=False,method="multi")

In [ ]:
#DROP THE TABLE IF EXISTS code is added to remove blank rows from table

%sql DROP TABLE IF EXISTS SPACEXTABLE;

In [ ]:
%sql create table SPACEXTABLE as select * from SPACEXTBL where Date is not null

### Display the names of the unique launch sites in the space mission

In [ ]:
%%sql
SELECT DISTINCT "Launch_Site"
FROM SPACEXTABLE ;

### Display 5 records where launch sites begin with the string 'CCA'

In [ ]:
# Run the query using pandas and read the result into a DataFrame
df = pd.read_sql_query("SELECT * FROM SPACEXTABLE WHERE Launch_Site LIKE 'CCA%' LIMIT 5", con)

# Display the result
df.head()

In [ ]:
%%sql
SELECT *
FROM SPACEXTABLE 
LIMIT 5;

In [ ]:
df.head()

### Display the total payload mass carried by boosters launched by NASA (CRS)

In [ ]:
%%sql
SELECT SUM("PAYLOAD_MASS__KG_") AS Total_Payload_Mass
FROM SPACEXTABLE
WHERE "Customer" = 'NASA (CRS)' ;

### Display average payload mass carried by booster version F9 v1.1

In [ ]:
%%sql
SELECT AVG("PAYLOAD_MASS__KG_") AS Average_Payload_Mass
FROM SPACEXTABLE
WHERE "Booster_Version" = 'F9 v1.1';

### List the date when the first successful landing outcome in ground pad was achieved

In [ ]:
%%sql
SELECT MIN("Date") AS First_Successful_Landing_Date
FROM SPACEXTABLE
WHERE "Mission_Outcome" = 'Success' AND "Landing_Outcome" = 'Success (ground pad)';

### List the names of the boosters which have success in drone ship and have payload mass greater than 4000 but less than 6000

In [ ]:
%%sql
SELECT DISTINCT "Booster_Version"
FROM SPACEXTABLE
WHERE "Mission_Outcome" = 'Success' 
  AND "Landing_Outcome" = 'Success (drone ship)'
  AND "PAYLOAD_MASS__KG_" > 4000 
  AND "PAYLOAD_MASS__KG_" < 6000;

### List the total number of successful and failure mission outcomes

In [ ]:
%%sql
SELECT "Mission_Outcome", COUNT(*) AS Outcome_Count
FROM SPACEXTABLE
GROUP BY "Mission_Outcome";

### List the names of the booster_versions which have carried the maximum payload mass. Use a subquery

In [ ]:
%%sql
SELECT "Booster_Version"
FROM SPACEXTABLE
WHERE "PAYLOAD_MASS__KG_" = (
    SELECT MAX("PAYLOAD_MASS__KG_")
    FROM SPACEXTABLE
);

### List the records which will display the month names, failure landing_outcomes in drone ship, booster versions, launch_site for the months in year 2015.

In [ ]:
%%sql
SELECT 
    CASE 
        WHEN SUBSTR("Date", 6, 2) = '01' THEN 'January'
        WHEN SUBSTR("Date", 6, 2) = '02' THEN 'February'
        WHEN SUBSTR("Date", 6, 2) = '03' THEN 'March'
        WHEN SUBSTR("Date", 6, 2) = '04' THEN 'April'
        WHEN SUBSTR("Date", 6, 2) = '05' THEN 'May'
        WHEN SUBSTR("Date", 6, 2) = '06' THEN 'June'
        WHEN SUBSTR("Date", 6, 2) = '07' THEN 'July'
        WHEN SUBSTR("Date", 6, 2) = '08' THEN 'August'
        WHEN SUBSTR("Date", 6, 2) = '09' THEN 'September'
        WHEN SUBSTR("Date", 6, 2) = '10' THEN 'October'
        WHEN SUBSTR("Date", 6, 2) = '11' THEN 'November'
        WHEN SUBSTR("Date", 6, 2) = '12' THEN 'December'
    END AS "Month_Name",
    "Landing_Outcome",
    "Booster_Version",
    "Launch_Site"
FROM SPACEXTABLE
WHERE "Landing_Outcome" = 'Failure (drone ship)'
  AND SUBSTR("Date", 0, 5) = '2015';


### Rank the count of landing outcomes (such as Failure (drone ship) or Success (ground pad)) between the date 2010-06-04 and 2017-03-20, in descending order.

In [ ]:
%%sql
SELECT "Landing_Outcome", COUNT(*) AS Outcome_Count
FROM SPACEXTABLE
WHERE "Date" BETWEEN '2010-06-04' AND '2017-03-20'
GROUP BY "Landing_Outcome"
ORDER BY Outcome_Count DESC;